# Strategy V1

In [1]:
%%capture
!pip install --upgrade binance-historical-data plotly requests tqdm -q


In [2]:
import os
import time
import warnings
from itertools import product

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from tqdm.auto import tqdm
from binance_historical_data import BinanceDataDumper

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [3]:
END_DATE = (pd.Timestamp.utcnow().normalize() - pd.Timedelta(days=1)).date()
START_DATE = (pd.Timestamp(END_DATE) - pd.DateOffset(years=5)).date()

DOWNLOAD_START = (pd.Timestamp(START_DATE) - pd.Timedelta(days=420)).date()

DATA_FREQUENCY = "4h"
MARKET_TYPE = "spot"
N_COINS = 10

TRADING_COST = 0.001   # 0.10% per one-way turnover
BAR_ANNUALIZATION = 365 * 6
DAILY_ANNUALIZATION = 365

SEARCH_MODE = "standard"   # standard / expanded
AUTO_RESCUE = True
RANDOM_SEED = 42

print(f"Backtest: {START_DATE} -> {END_DATE}")
print(f"Download incl. warm-up: {DOWNLOAD_START} -> {END_DATE}")
print(f"Timeframe: {DATA_FREQUENCY}")
print(f"Search mode: {SEARCH_MODE}")


Backtest: 2021-08-19 -> 2026-08-19
Download incl. warm-up: 2020-06-25 -> 2026-08-19
Timeframe: 4h
Search mode: standard


In [4]:
def create_data_dumper(market_type="spot", data_frequency="4h"):
    return BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class=market_type,
        data_type="klines",
        data_frequency=data_frequency,
    )

def get_list_all_trading_pairs(market_type="spot"):
    return create_data_dumper(market_type, DATA_FREQUENCY).get_list_all_trading_pairs()

def find_first_data_date(ticker, market_type="spot", data_frequency="4h"):
    dumper = create_data_dumper(market_type, data_frequency)
    return dumper.get_min_start_date_for_ticker(ticker)

def download_ticker(ticker, date_start, date_end, data_frequency="4h", market_type="spot"):
    dumper = create_data_dumper(market_type, data_frequency)
    dumper.dump_data(
        tickers=ticker,
        date_start=pd.Timestamp(date_start).date(),
        date_end=pd.Timestamp(date_end).date(),
        is_to_update_existing=False,
    )

COLS = [
    "open_time", "Open", "High", "Low", "Close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
]

def _parse_binance_timestamp(series):
    x = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    ms = x.notna() & (x.abs() < 1e15)
    us = x.notna() & ~ms

    if ms.any():
        out.loc[ms] = pd.to_datetime(x.loc[ms], unit="ms", errors="coerce")
    if us.any():
        out.loc[us] = pd.to_datetime(x.loc[us], unit="us", errors="coerce")
    return out

def read_csv_file(file_path):
    df = pd.read_csv(file_path, header=None)

    if df.shape[1] < 12:
        raise ValueError(f"Unexpected Binance CSV format: {file_path}")

    df = df.iloc[:, :12]
    df.columns = COLS

    df["open_time"] = _parse_binance_timestamp(df["open_time"])
    df["close_time"] = _parse_binance_timestamp(df["close_time"])

    for c in ["Open", "High", "Low", "Close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["open_time", "Open", "High", "Low", "Close"])
    return df

def get_csv_files(directory):
    if not os.path.exists(directory):
        return []

    return sorted(
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.endswith(".csv")
    )

def process_csv_files(ticker, data_frequency="4h", market_type="spot"):
    daily_path = os.path.join(
        os.getcwd(), market_type, "daily", "klines", ticker, data_frequency
    )
    monthly_path = os.path.join(
        os.getcwd(), market_type, "monthly", "klines", ticker, data_frequency
    )

    files = get_csv_files(daily_path) + get_csv_files(monthly_path)

    if not files:
        return None

    data = pd.concat([read_csv_file(f) for f in files], ignore_index=True)
    data = data.sort_values("open_time")
    data = data.drop_duplicates(subset=["open_time"], keep="last")
    return data


In [5]:
%%capture
STABLE_SYMBOLS = {
    "USDT", "USDC", "USDS", "DAI", "FDUSD", "TUSD", "USDE", "USD1", "USDD",
    "USDF", "USDG", "PYUSD", "RLUSD", "BUSD", "GUSD", "LUSD", "FRAX", "USDP",
    "SUSDS", "USDY", "BFUSD", "EURC", "EURCV", "USDX"
}

def get_market_cap_ranking(per_page=100):
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": per_page,
        "page": 1,
        "sparkline": "false",
    }
    headers = {"User-Agent": "Mozilla/5.0"}

    last_error = None

    for attempt in range(4):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            r.raise_for_status()
            raw = r.json()

            rows = []
            for x in raw:
                rows.append({
                    "rank": x.get("market_cap_rank"),
                    "name": x.get("name"),
                    "symbol": str(x.get("symbol", "")).upper(),
                    "market_cap": x.get("market_cap"),
                })

            df = pd.DataFrame(rows)
            df = df.dropna(subset=["rank", "symbol"])
            df = df.sort_values("rank").drop_duplicates("symbol", keep="first")
            return df

        except Exception as e:
            last_error = e
            time.sleep(2 ** attempt)

    raise RuntimeError(
        "CoinGecko market-cap ranking failed after retries. "
        "Rerun this cell later. Error: " + repr(last_error)
    )

market_rank = get_market_cap_ranking(100)
all_pairs = set(get_list_all_trading_pairs(MARKET_TYPE))

eligible = []

for _, row in market_rank.iterrows():
    sym = row["symbol"]

    if sym in STABLE_SYMBOLS:
        continue

    ticker = f"{sym}USDT"

    if ticker not in all_pairs:
        continue

    try:
        first_date = pd.Timestamp(
            find_first_data_date(ticker, MARKET_TYPE, DATA_FREQUENCY)
        ).date()
    except Exception:
        continue

    if first_date > (pd.Timestamp(START_DATE) + pd.Timedelta(days=7)).date():
        continue

    eligible.append({
        **row.to_dict(),
        "ticker": ticker,
        "first_date": first_date,
    })

    if len(eligible) == N_COINS:
        break

universe_df = pd.DataFrame(eligible)

if len(universe_df) < N_COINS:
    raise RuntimeError(
        f"Only {len(universe_df)} assets satisfy market-cap/Binance/history requirements."
    )

TICKERS = universe_df["ticker"].tolist()

print("Selected trading universe:")
display(
    universe_df[
        ["rank", "name", "symbol", "ticker", "first_date", "market_cap"]
    ].style.format({"market_cap": "{:,.0f}"})
)

DATA_TICKERS = list(dict.fromkeys(TICKERS + ["BTCUSDT"]))


In [6]:
%%capture
for ticker in tqdm(DATA_TICKERS, desc="Downloading Binance 4H data"):
    print(f"\n{ticker}")

    download_ticker(
        ticker=ticker,
        date_start=DOWNLOAD_START,
        date_end=END_DATE,
        data_frequency=DATA_FREQUENCY,
        market_type=MARKET_TYPE,
    )

def to_4h_ohlcv(raw_df):
    x = raw_df.copy()

    x = x[
        (x["open_time"].dt.date >= DOWNLOAD_START)
        & (x["open_time"].dt.date <= END_DATE)
    ].copy()

    x = x.set_index("open_time").sort_index()

    x = x[["Open", "High", "Low", "Close", "volume"]].apply(
        pd.to_numeric,
        errors="coerce",
    )

    x = x.dropna(subset=["Open", "High", "Low", "Close"])
    return x

all_market = {}
coverage_rows = []

expected_bars = max(
    1,
    int(
        (
            (pd.Timestamp(END_DATE) + pd.Timedelta(days=1))
            - pd.Timestamp(DOWNLOAD_START)
        ).total_seconds()
        / (4 * 3600)
    )
)

for ticker in DATA_TICKERS:
    raw = process_csv_files(ticker, DATA_FREQUENCY, MARKET_TYPE)

    if raw is None or raw.empty:
        raise RuntimeError(f"No Binance data loaded for {ticker}")

    symbol = ticker[:-4]
    df = to_4h_ohlcv(raw)
    all_market[symbol] = df

    coverage_rows.append({
        "symbol": symbol,
        "bars": len(df),
        "coverage_%": 100 * len(df) / expected_bars,
        "first_bar": df.index.min(),
        "last_bar": df.index.max(),
    })

coverage_df = pd.DataFrame(coverage_rows)

print("Data coverage:")
display(
    coverage_df.style.format({"coverage_%": "{:.2f}%"})
)

bad = coverage_df[coverage_df["coverage_%"] < 95]

if len(bad):
    raise RuntimeError(
        "At least one asset has <95% expected 4H coverage. "
        "Inspect the table before backtesting."
    )

trade_symbols = [t[:-4] for t in TICKERS]

common_index = None

for s in list(dict.fromkeys(trade_symbols + ["BTC"])):
    idx = all_market[s].index
    common_index = idx if common_index is None else common_index.intersection(idx)

common_index = common_index.sort_values()

market = {
    s: all_market[s].reindex(common_index)
    for s in trade_symbols
}

btc_market = all_market["BTC"].reindex(common_index)

print("Common bars:", len(common_index))
print("Common period:", common_index.min(), "->", common_index.max())


---> Found overall tickers: 627
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 4h
---> Start Date: 20200625
---> End Date: 20260819
Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 74 months 18 days
---> For BTCUSD new data saved for: 8 months 18 days
---> Found overall tickers: 627
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 4h
---> Start Date: 20200625
---> End Date: 20260819
Tried to dump data for 2 tickers:
---> For ETHUSDT new data saved for: 74 months 18 days
---> For ETHUSD new data saved for: 5 months 18 days
---> Found overall tickers: 627
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 4h
---> Start Date: 20200625
---> End Date: 20

In [7]:
def performance_stats(equity):
    eq_4h = pd.Series(equity).dropna().astype(float)

    if len(eq_4h) < 30:
        return {
            "Sharpe": np.nan,
            "CAGR": np.nan,
            "Vol": np.nan,
            "MDD": np.nan,
            "Calmar": np.nan,
            "Total Return": np.nan,
        }

    eq_4h = eq_4h / eq_4h.iloc[0]

    dd_4h = eq_4h / eq_4h.cummax() - 1.0
    mdd = dd_4h.min()

    eq_daily = eq_4h.resample("1D").last().dropna()

    if len(eq_daily) < 20:
        return {
            "Sharpe": np.nan,
            "CAGR": np.nan,
            "Vol": np.nan,
            "MDD": mdd,
            "Calmar": np.nan,
            "Total Return": eq_4h.iloc[-1] - 1.0,
        }

    daily_ret = eq_daily.pct_change().fillna(0.0)
    std = daily_ret.std(ddof=0)

    days = max(
        1.0,
        (eq_daily.index[-1] - eq_daily.index[0]).total_seconds() / 86400.0,
    )
    years = days / 365.0

    cagr = (
        eq_daily.iloc[-1] ** (1.0 / years) - 1.0
        if years > 0 and eq_daily.iloc[-1] > 0
        else np.nan
    )

    vol = std * np.sqrt(DAILY_ANNUALIZATION)

    sharpe = (
        daily_ret.mean() / std * np.sqrt(DAILY_ANNUALIZATION)
        if std > 0
        else np.nan
    )

    calmar = (
        cagr / abs(mdd)
        if pd.notna(cagr) and mdd < 0
        else np.nan
    )

    return {
        "Sharpe": sharpe,
        "CAGR": cagr,
        "Vol": vol,
        "MDD": mdd,
        "Calmar": calmar,
        "Total Return": eq_4h.iloc[-1] - 1.0,
    }


def compute_asset_indicators(
    df,
    breakout_window,
    trend_fast_ema,
    trend_slow_ema,
    atr_window=14,
    vol_window=42,
):
    x = df.copy()

    prev_close = x["Close"].shift(1)

    tr = pd.concat(
        [
            x["High"] - x["Low"],
            (x["High"] - prev_close).abs(),
            (x["Low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    x["ATR"] = tr.ewm(
        alpha=1 / atr_window,
        adjust=False,
        min_periods=atr_window,
    ).mean()

    x["EMA_FAST"] = x["Close"].ewm(
        span=trend_fast_ema,
        adjust=False,
        min_periods=trend_fast_ema,
    ).mean()

    x["EMA_SLOW"] = x["Close"].ewm(
        span=trend_slow_ema,
        adjust=False,
        min_periods=trend_slow_ema,
    ).mean()

    x["RES"] = (
        x["High"]
        .shift(1)
        .rolling(breakout_window)
        .max()
    )

    x["VOL_MED"] = (
        x["volume"]
        .shift(1)
        .rolling(breakout_window)
        .median()
    )

    r = x["Close"].pct_change()

    x["ANN_VOL"] = (
        r.rolling(vol_window).std(ddof=0)
        * np.sqrt(BAR_ANNUALIZATION)
    )

    x["MOM_21D"] = x["Close"].pct_change(21 * 6)
    x["MOM_63D"] = x["Close"].pct_change(63 * 6)
    x["RS_RAW"] = 0.65 * x["MOM_21D"] + 0.35 * x["MOM_63D"]

    x["TREND_OK"] = (
        (x["Close"] > x["EMA_FAST"])
        & (x["EMA_FAST"] > x["EMA_SLOW"])
        & (x["EMA_SLOW"] > x["EMA_SLOW"].shift(6))
    )

    return x


def compute_btc_risk_scale(
    btc_df,
    btc_fast_days,
    btc_slow_days,
    btc_mom_days,
):
    daily = (
        btc_df[["Close"]]
        .resample("1D")
        .last()
        .dropna()
    )

    fast = daily["Close"].ewm(
        span=btc_fast_days,
        adjust=False,
        min_periods=btc_fast_days,
    ).mean()

    slow = daily["Close"].ewm(
        span=btc_slow_days,
        adjust=False,
        min_periods=btc_slow_days,
    ).mean()

    mom = daily["Close"].pct_change(btc_mom_days)

    bullish = (
        (daily["Close"] > slow)
        & (fast > slow)
        & (mom > 0)
    )

    bearish = (
        (daily["Close"] < slow)
        & (fast < slow)
        & (mom < 0)
    )

    scale = pd.Series(
        0.55,
        index=daily.index,
        dtype=float,
    )

    scale.loc[bullish] = 1.0
    scale.loc[bearish] = 0.0

    scale = scale.shift(1).fillna(0.0)

    return (
        scale
        .reindex(btc_df.index, method="ffill")
        .fillna(0.0)
    )


def drawdown_risk_scale(
    prior_equity_values,
    lookback_bars,
    dd_level_1,
    dd_level_2,
    dd_level_3,
    dd_hard,
):
    if len(prior_equity_values) < 2:
        return 1.0, 0.0

    prior_eq = float(prior_equity_values[-1])

    start = max(
        0,
        len(prior_equity_values) - int(lookback_bars),
    )

    window = prior_equity_values[start:]
    peak = max(window) if window else prior_eq

    dd = (
        prior_eq / peak - 1.0
        if peak > 0
        else 0.0
    )

    if dd <= -dd_hard:
        scale = 0.0
    elif dd <= -dd_level_3:
        scale = 0.35
    elif dd <= -dd_level_2:
        scale = 0.60
    elif dd <= -dd_level_1:
        scale = 0.80
    else:
        scale = 1.0

    return scale, dd


def run_strategy_v1(
    market,
    btc_market,

    breakout_window=48,
    breakout_buffer_atr=0.15,
    breakout_entry_fraction=0.45,
    retest_window=6,
    retest_atr=0.65,
    retest_close_buffer_atr=0.00,
    volume_mult=1.10,
    min_rs_percentile=0.60,

    trend_fast_ema=60,
    trend_slow_ema=160,

    btc_fast_days=40,
    btc_slow_days=140,
    btc_mom_days=45,

    breakout_close_location=0.60,
    breakout_body_atr=0.08,
    max_breakout_extension_atr=3.00,

    initial_stop_atr=2.20,
    structural_stop_buffer_atr=0.25,
    trailing_stop_atr=3.50,
    trail_activation_r=1.10,
    break_even_r=0.90,
    profit_lock_r=2.75,
    profit_lock_trail_factor=0.78,
    time_stop_bars=42,
    time_stop_min_r=0.05,

    max_positions=4,
    max_position_weight=0.35,
    position_vol_target=0.24,
    risk_per_trade=0.020,
    max_gross_exposure=1.00,

    dd_lookback_bars=540,
    dd_level_1=0.12,
    dd_level_2=0.18,
    dd_level_3=0.24,
    dd_hard=0.275,

    atr_window=14,
    vol_window=42,
    fee=0.001,
    start_date=None,
):
    symbols = list(market.keys())
    n_assets = len(symbols)

    prepared = {
        s: compute_asset_indicators(
            market[s],
            breakout_window=breakout_window,
            trend_fast_ema=trend_fast_ema,
            trend_slow_ema=trend_slow_ema,
            atr_window=atr_window,
            vol_window=vol_window,
        )
        for s in symbols
    }

    idx = prepared[symbols[0]].index

    for s in symbols[1:]:
        idx = idx.intersection(prepared[s].index)

    idx = idx.intersection(btc_market.index).sort_values()

    btc_scale = (
        compute_btc_risk_scale(
            btc_market.reindex(idx),
            btc_fast_days=btc_fast_days,
            btc_slow_days=btc_slow_days,
            btc_mom_days=btc_mom_days,
        )
        .reindex(idx)
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    fields = [
        "Open", "High", "Low", "Close", "volume",
        "ATR", "EMA_FAST", "EMA_SLOW",
        "RES", "VOL_MED", "ANN_VOL", "RS_RAW",
    ]

    A = {}

    for field in fields:
        A[field] = np.column_stack(
            [
                pd.to_numeric(
                    prepared[s].reindex(idx)[field],
                    errors="coerce",
                ).to_numpy(dtype=float)
                for s in symbols
            ]
        )

    trend_ok = np.column_stack(
        [
            prepared[s]
            .reindex(idx)["TREND_OK"]
            .fillna(False)
            .to_numpy(dtype=bool)
            for s in symbols
        ]
    )

    rs_pct = (
        pd.DataFrame(
            A["RS_RAW"],
            index=idx,
            columns=symbols,
        )
        .rank(axis=1, pct=True)
        .to_numpy(dtype=float)
    )

    start_ts = (
        pd.Timestamp(start_date)
        if start_date is not None
        else idx[0]
    )

    first_trade_i = int(idx.searchsorted(start_ts))

    warmup = max(
        breakout_window + 2,
        trend_slow_ema + 10,
        63 * 6 + 10,
        atr_window + 2,
        vol_window + 2,
    )

    cash = 1.0

    positions = {}

    pending_retest = {}

    next_starters = []
    next_adds = []
    next_exits = {}

    trades = []

    equity_vals = []
    exposure_vals = []
    btc_scale_vals = []
    dd_scale_vals = []
    rolling_dd_vals = []
    equity_idx = []

    def mark_open_equity(i):
        eq = cash
        position_value = 0.0

        for j, pos in positions.items():
            px = A["Open"][i, j]

            if np.isfinite(px) and px > 0:
                value = pos["units"] * px
                eq += value
                position_value += value

        return eq, position_value

    def target_position_weight(
        px,
        atr,
        ann_vol,
        equity_open,
    ):
        stop_distance = max(
            initial_stop_atr * atr,
            px * 0.0025,
        )

        stop_pct = stop_distance / px

        vol_weight = (
            position_vol_target
            / max(ann_vol, 1e-6)
        )

        risk_weight = (
            risk_per_trade
            / max(stop_pct, 1e-6)
        )

        return min(
            max_position_weight,
            vol_weight,
            risk_weight,
        )

    for i in range(len(idx)):
        ts = idx[i]

        if i < first_trade_i:
            continue

        current_btc_scale = float(btc_scale[i])

        dd_scale, prior_rolling_dd = drawdown_risk_scale(
            equity_vals,
            dd_lookback_bars,
            dd_level_1,
            dd_level_2,
            dd_level_3,
            dd_hard,
        )

        new_risk_scale = (
            current_btc_scale
            * dd_scale
        )

        for j, reason in list(next_exits.items()):
            if j not in positions:
                continue

            px = A["Open"][i, j]

            if not np.isfinite(px) or px <= 0:
                continue

            pos = positions.pop(j)
            pending_retest.pop(j, None)

            proceeds = (
                pos["units"] * px * (1 - fee)
            )

            cash += proceeds
            pnl = proceeds - pos["total_cost"]

            trades.append({
                "symbol": symbols[j],
                "entry_time": pos["entry_time"],
                "exit_time": ts,
                "entry_price": pos["avg_entry"],
                "exit_price": px,
                "exit_reason": reason,
                "pnl": pnl,
                "return_pct": (
                    pnl / pos["total_cost"]
                    if pos["total_cost"] > 0
                    else np.nan
                ),
                "bars_held": i - pos["entry_i"],
                "added_on_retest": bool(pos["added_on_retest"]),
                "initial_risk_pct": (
                    pos["initial_risk"]
                    / pos["avg_entry"]
                    if pos["avg_entry"] > 0
                    else np.nan
                ),
            })

        next_exits = {}

        if next_starters and new_risk_scale > 0:

            slots = max(
                0,
                max_positions - len(positions),
            )

            starters = [
                sig
                for sig in next_starters
                if sig["j"] not in positions
            ]

            starters = sorted(
                starters,
                key=lambda z: z["score"],
                reverse=True,
            )[:slots]

            if starters:

                equity_open, current_pos_value = mark_open_equity(i)

                effective_gross_cap = min(
                    1.0,
                    max_gross_exposure * new_risk_scale,
                )

                gross_budget = max(
                    0.0,
                    effective_gross_cap * equity_open
                    - current_pos_value,
                )

                proposed = []

                for sig in starters:
                    j = sig["j"]
                    px = A["Open"][i, j]
                    atr = sig["atr"]
                    ann_vol = sig["ann_vol"]

                    if (
                        not np.isfinite(px)
                        or px <= 0
                        or not np.isfinite(atr)
                        or atr <= 0
                        or not np.isfinite(ann_vol)
                        or ann_vol <= 0
                    ):
                        continue

                    gap_extension = (
                        px - sig["breakout_level"]
                    ) / max(atr, 1e-9)

                    if gap_extension > 1.75:
                        continue

                    full_target_weight = target_position_weight(
                        px=px,
                        atr=atr,
                        ann_vol=ann_vol,
                        equity_open=equity_open,
                    )

                    starter_weight = (
                        full_target_weight
                        * breakout_entry_fraction
                    )

                    desired_notional = (
                        starter_weight * equity_open
                    )

                    proposed.append({
                        "sig": sig,
                        "px": px,
                        "atr": atr,
                        "full_target_weight": full_target_weight,
                        "desired_notional": desired_notional,
                    })

                total_desired = sum(
                    x["desired_notional"]
                    for x in proposed
                )

                scale = 1.0

                if (
                    total_desired > gross_budget
                    and total_desired > 0
                ):
                    scale = min(
                        scale,
                        gross_budget / total_desired,
                    )

                total_cost = sum(
                    x["desired_notional"]
                    * scale
                    * (1 + fee)
                    for x in proposed
                )

                if total_cost > cash and total_cost > 0:
                    scale *= cash / total_cost

                for order in proposed:
                    sig = order["sig"]
                    j = sig["j"]
                    px = order["px"]
                    atr = order["atr"]

                    notional = (
                        order["desired_notional"]
                        * scale
                    )

                    notional = min(
                        notional,
                        cash / (1 + fee),
                    )

                    if notional <= 1e-10:
                        continue

                    units = notional / px
                    cost = notional * (1 + fee)
                    cash -= cost

                    structural_stop = (
                        sig["breakout_level"]
                        - structural_stop_buffer_atr * atr
                    )

                    vol_stop = (
                        px - initial_stop_atr * atr
                    )

                    initial_stop = max(
                        structural_stop,
                        vol_stop,
                    )

                    initial_stop = min(
                        initial_stop,
                        px * 0.9975,
                    )

                    initial_risk = max(
                        px - initial_stop,
                        px * 0.0025,
                    )

                    positions[j] = {
                        "units": units,
                        "avg_entry": px,
                        "total_cost": cost,
                        "entry_time": ts,
                        "entry_i": i,
                        "breakout_level": sig["breakout_level"],
                        "stop": initial_stop,
                        "initial_risk": initial_risk,
                        "peak_high": px,
                        "peak_close": px,
                        "trail_active": False,
                        "added_on_retest": False,
                        "full_target_weight": order["full_target_weight"],
                    }

                    pending_retest[j] = {
                        "level": sig["breakout_level"],
                        "breakout_i": sig["breakout_i"],
                        "expiry_i": sig["breakout_i"] + retest_window,
                    }

        next_starters = []

        if next_adds and new_risk_scale > 0:

            equity_open, current_pos_value = mark_open_equity(i)

            effective_gross_cap = min(
                1.0,
                max_gross_exposure * new_risk_scale,
            )

            gross_budget = max(
                0.0,
                effective_gross_cap * equity_open
                - current_pos_value,
            )

            adds = sorted(
                next_adds,
                key=lambda z: z["score"],
                reverse=True,
            )

            proposed_adds = []

            for sig in adds:
                j = sig["j"]

                if j not in positions:
                    continue

                pos = positions[j]

                if pos["added_on_retest"]:
                    continue

                px = A["Open"][i, j]
                atr = sig["atr"]
                ann_vol = sig["ann_vol"]

                if (
                    not np.isfinite(px)
                    or px <= 0
                    or not np.isfinite(atr)
                    or atr <= 0
                    or not np.isfinite(ann_vol)
                    or ann_vol <= 0
                ):
                    continue

                current_value = (
                    pos["units"] * px
                )

                full_target_weight = target_position_weight(
                    px=px,
                    atr=atr,
                    ann_vol=ann_vol,
                    equity_open=equity_open,
                )

                desired_full_value = (
                    full_target_weight * equity_open
                )

                add_notional = max(
                    0.0,
                    desired_full_value - current_value,
                )

                proposed_adds.append({
                    "sig": sig,
                    "px": px,
                    "add_notional": add_notional,
                })

            total_desired = sum(
                x["add_notional"]
                for x in proposed_adds
            )

            scale = 1.0

            if (
                total_desired > gross_budget
                and total_desired > 0
            ):
                scale = min(
                    scale,
                    gross_budget / total_desired,
                )

            total_cost = sum(
                x["add_notional"]
                * scale
                * (1 + fee)
                for x in proposed_adds
            )

            if total_cost > cash and total_cost > 0:
                scale *= cash / total_cost

            for order in proposed_adds:
                sig = order["sig"]
                j = sig["j"]

                if j not in positions:
                    continue

                px = order["px"]
                add_notional = (
                    order["add_notional"]
                    * scale
                )

                add_notional = min(
                    add_notional,
                    cash / (1 + fee),
                )

                pos = positions[j]

                if add_notional > 1e-10:
                    add_units = add_notional / px
                    add_cost = add_notional * (1 + fee)

                    old_units = pos["units"]
                    new_units = old_units + add_units

                    pos["avg_entry"] = (
                        old_units * pos["avg_entry"]
                        + add_units * px
                    ) / new_units

                    pos["units"] = new_units
                    pos["total_cost"] += add_cost
                    cash -= add_cost

                    retest_structural_stop = (
                        sig["retest_low"]
                        - structural_stop_buffer_atr * sig["atr"]
                    )

                    pos["stop"] = max(
                        pos["stop"],
                        retest_structural_stop,
                    )

                    pos["initial_risk"] = max(
                        pos["avg_entry"] - pos["stop"],
                        pos["avg_entry"] * 0.0025,
                    )

                pos["added_on_retest"] = True
                pending_retest.pop(j, None)

        next_adds = []

        for j in list(positions.keys()):
            pos = positions[j]

            op = A["Open"][i, j]
            low = A["Low"][i, j]
            stop = pos["stop"]

            if not (
                np.isfinite(op)
                and np.isfinite(low)
                and np.isfinite(stop)
            ):
                continue

            exit_px = None

            if op <= stop:
                exit_px = op
            elif low <= stop:
                exit_px = stop

            if exit_px is not None and exit_px > 0:

                pos = positions.pop(j)
                pending_retest.pop(j, None)

                proceeds = (
                    pos["units"]
                    * exit_px
                    * (1 - fee)
                )

                cash += proceeds
                pnl = proceeds - pos["total_cost"]

                trades.append({
                    "symbol": symbols[j],
                    "entry_time": pos["entry_time"],
                    "exit_time": ts,
                    "entry_price": pos["avg_entry"],
                    "exit_price": exit_px,
                    "exit_reason": "atr_stop",
                    "pnl": pnl,
                    "return_pct": (
                        pnl / pos["total_cost"]
                        if pos["total_cost"] > 0
                        else np.nan
                    ),
                    "bars_held": i - pos["entry_i"],
                    "added_on_retest": bool(pos["added_on_retest"]),
                    "initial_risk_pct": (
                        pos["initial_risk"] / pos["avg_entry"]
                        if pos["avg_entry"] > 0
                        else np.nan
                    ),
                })

        for j, pos in list(positions.items()):

            high = A["High"][i, j]
            close = A["Close"][i, j]
            atr = A["ATR"][i, j]
            ema_fast = A["EMA_FAST"][i, j]
            ema_slow = A["EMA_SLOW"][i, j]

            if np.isfinite(high):
                pos["peak_high"] = max(
                    pos["peak_high"],
                    high,
                )

            if np.isfinite(close):
                pos["peak_close"] = max(
                    pos["peak_close"],
                    close,
                )

            if not (
                np.isfinite(close)
                and np.isfinite(atr)
                and atr > 0
            ):
                continue

            mfe = (
                pos["peak_high"]
                - pos["avg_entry"]
            )

            r_multiple = (
                mfe
                / max(pos["initial_risk"], 1e-9)
            )

            if r_multiple >= break_even_r:
                fee_buffered_be = (
                    pos["avg_entry"] * (1 + 2 * fee)
                )

                pos["stop"] = max(
                    pos["stop"],
                    fee_buffered_be,
                )

            if r_multiple >= trail_activation_r:
                pos["trail_active"] = True

            if pos["trail_active"]:

                dynamic_trailing_atr = trailing_stop_atr

                if r_multiple >= profit_lock_r:
                    dynamic_trailing_atr *= (
                        profit_lock_trail_factor
                    )

                trail = (
                    pos["peak_close"]
                    - dynamic_trailing_atr * atr
                )

                pos["stop"] = max(
                    pos["stop"],
                    trail,
                )

            held_bars = i - pos["entry_i"]

            current_r = (
                close - pos["avg_entry"]
            ) / max(
                pos["initial_risk"],
                1e-9,
            )


            if (
                np.isfinite(ema_slow)
                and close < ema_slow
            ):
                next_exits[j] = "slow_trend_exit"
                continue

            if (
                np.isfinite(ema_fast)
                and close < ema_fast
                and current_r < 0.75
            ):
                next_exits[j] = "fast_trend_exit"
                continue

            if (
                close
                < pos["breakout_level"]
                - 0.30 * atr
                and current_r < 0.50
            ):
                next_exits[j] = "failed_breakout"
                continue

            if (
                time_stop_bars is not None
                and time_stop_bars > 0
                and held_bars >= time_stop_bars
                and current_r < time_stop_min_r
            ):
                next_exits[j] = "time_stop"

        starter_signals = []
        add_signals = []

        if i >= max(first_trade_i, warmup):

            for j in list(pending_retest.keys()):

                if j not in positions:
                    pending_retest.pop(j, None)
                    continue

                p = pending_retest[j]

                if i > p["expiry_i"]:
                    pending_retest.pop(j, None)
                    continue

                if i <= p["breakout_i"]:
                    continue

                op = A["Open"][i, j]
                high = A["High"][i, j]
                low = A["Low"][i, j]
                close = A["Close"][i, j]
                atr = A["ATR"][i, j]
                ann_vol = A["ANN_VOL"][i, j]

                if not np.all(
                    np.isfinite(
                        [op, high, low, close, atr, ann_vol]
                    )
                ):
                    continue

                if atr <= 0 or ann_vol <= 0:
                    continue

                if (
                    low
                    < p["level"]
                    - retest_atr * atr
                ):
                    pending_retest.pop(j, None)
                    continue

                candle_range = max(
                    high - low,
                    1e-9,
                )

                close_location = (
                    close - low
                ) / candle_range

                touched = (
                    low
                    <= p["level"]
                    + retest_atr * atr
                )

                reclaimed = (
                    close
                    >= p["level"]
                    + retest_close_buffer_atr * atr
                )

                bullish = (
                    close > op
                    and close_location >= 0.52
                )

                if (
                    touched
                    and reclaimed
                    and bullish
                    and trend_ok[i, j]
                    and rs_pct[i, j] >= max(
                        0.50,
                        min_rs_percentile - 0.10,
                    )
                    and new_risk_scale > 0
                ):
                    retest_quality = max(
                        0.0,
                        1.0
                        - abs(low - p["level"])
                        / max(
                            retest_atr * atr,
                            1e-6,
                        ),
                    )

                    add_signals.append({
                        "j": j,
                        "score": (
                            0.60 * retest_quality
                            + 0.40 * rs_pct[i, j]
                        ),
                        "atr": atr,
                        "ann_vol": ann_vol,
                        "breakout_level": p["level"],
                        "retest_low": low,
                    })

            if new_risk_scale > 0:

                for j in range(n_assets):

                    if j in positions:
                        continue

                    op = A["Open"][i, j]
                    high = A["High"][i, j]
                    low = A["Low"][i, j]
                    close = A["Close"][i, j]
                    vol_now = A["volume"][i, j]
                    atr = A["ATR"][i, j]
                    res = A["RES"][i, j]
                    vol_med = A["VOL_MED"][i, j]
                    ann_vol = A["ANN_VOL"][i, j]
                    rs = rs_pct[i, j]

                    vals = [
                        op, high, low, close,
                        vol_now, atr, res,
                        vol_med, ann_vol, rs,
                    ]

                    if (
                        not np.all(np.isfinite(vals))
                        or atr <= 0
                        or ann_vol <= 0
                    ):
                        continue

                    if not trend_ok[i, j]:
                        continue

                    if rs < min_rs_percentile:
                        continue

                    candle_range = max(
                        high - low,
                        1e-9,
                    )

                    close_location = (
                        close - low
                    ) / candle_range

                    body_atr = (
                        close - op
                    ) / atr

                    volume_ratio = (
                        vol_now
                        / max(vol_med, 1e-12)
                    )

                    breakout_distance = (
                        close - res
                    ) / max(atr, 1e-9)

                    extension = (
                        high - res
                    ) / max(atr, 1e-9)

                    breakout_ok = (
                        close
                        > res
                        + breakout_buffer_atr * atr
                    )

                    candle_ok = (
                        close_location >= breakout_close_location
                        and body_atr >= breakout_body_atr
                    )

                    volume_ok = (
                        volume_ratio >= volume_mult
                    )

                    not_exhausted = (
                        extension
                        <= max_breakout_extension_atr
                    )

                    if (
                        breakout_ok
                        and candle_ok
                        and volume_ok
                        and not_exhausted
                    ):
                        starter_signals.append({
                            "j": j,
                            "score": (
                                0.45 * breakout_distance
                                + 0.25 * min(
                                    volume_ratio,
                                    3.0,
                                )
                                + 0.15 * close_location
                                + 0.15 * rs
                            ),
                            "atr": atr,
                            "ann_vol": ann_vol,
                            "breakout_level": res,
                            "breakout_i": i,
                        })

        if starter_signals:

            dedup = {}

            for sig in starter_signals:
                j = sig["j"]

                if (
                    j not in dedup
                    or sig["score"] > dedup[j]["score"]
                ):
                    dedup[j] = sig

            next_starters = sorted(
                dedup.values(),
                key=lambda z: z["score"],
                reverse=True,
            )

        if add_signals:

            dedup = {}

            for sig in add_signals:
                j = sig["j"]

                if (
                    j not in dedup
                    or sig["score"] > dedup[j]["score"]
                ):
                    dedup[j] = sig

            next_adds = sorted(
                dedup.values(),
                key=lambda z: z["score"],
                reverse=True,
            )

        equity = cash
        gross_exposure = 0.0

        for j, pos in positions.items():
            px = A["Close"][i, j]

            if np.isfinite(px) and px > 0:
                value = pos["units"] * px
                equity += value
                gross_exposure += value

        equity_idx.append(ts)
        equity_vals.append(equity)

        exposure_vals.append(
            gross_exposure / equity
            if equity > 0
            else 0.0
        )

        btc_scale_vals.append(
            current_btc_scale
        )

        dd_scale_vals.append(
            dd_scale
        )

        rolling_dd_vals.append(
            prior_rolling_dd
        )

    equity = pd.Series(
        equity_vals,
        index=pd.DatetimeIndex(equity_idx),
        name="equity",
    )

    if len(equity):
        equity = equity / equity.iloc[0]

    return {
        "equity": equity,
        "returns": equity.pct_change().fillna(0.0),
        "exposure": pd.Series(
            exposure_vals,
            index=equity.index,
            name="gross_exposure",
        ),
        "trades": pd.DataFrame(trades),
        "stats": performance_stats(equity),
        "btc_scale": pd.Series(
            btc_scale_vals,
            index=equity.index,
            name="btc_new_risk_scale",
        ),
        "dd_scale": pd.Series(
            dd_scale_vals,
            index=equity.index,
            name="drawdown_new_risk_scale",
        ),
        "rolling_dd": pd.Series(
            rolling_dd_vals,
            index=equity.index,
            name="rolling_drawdown",
        ),
    }


In [8]:
%%capture
base_space = {
    "breakout_window": [30, 42, 54, 72],
    "breakout_buffer_atr": [0.05, 0.12, 0.20, 0.30],
    "breakout_entry_fraction": [0.35, 0.45, 0.55, 0.65],
    "retest_window": [3, 6, 9],
    "retest_atr": [0.45, 0.65, 0.85],
    "volume_mult": [0.95, 1.10, 1.25, 1.40],
    "min_rs_percentile": [0.50, 0.60, 0.70],
    "trend_fast_ema": [40, 60, 80],
    "trend_slow_ema": [120, 160, 200],
    "btc_fast_days": [30, 40, 50],
    "btc_slow_days": [100, 140, 180],
    "btc_mom_days": [30, 45, 60],
    "initial_stop_atr": [1.80, 2.20, 2.60],
    "trailing_stop_atr": [3.00, 3.50, 4.00],
    "trail_activation_r": [0.90, 1.10, 1.35],
    "break_even_r": [0.75, 0.90, 1.10],
    "time_stop_bars": [30, 42, 60],
    "max_positions": [3, 4, 5],
    "position_vol_target": [0.20, 0.24, 0.28],
    "risk_per_trade": [0.016, 0.020, 0.024],
    "max_gross_exposure": [0.85, 1.00],
}

fixed_params = {
    "retest_close_buffer_atr": 0.00,
    "breakout_close_location": 0.60,
    "breakout_body_atr": 0.08,
    "max_breakout_extension_atr": 3.00,
    "structural_stop_buffer_atr": 0.25,
    "profit_lock_r": 2.75,
    "profit_lock_trail_factor": 0.78,
    "time_stop_min_r": 0.05,
    "max_position_weight": 0.35,
    "dd_lookback_bars": 540,
    "dd_level_1": 0.12,
    "dd_level_2": 0.18,
    "dd_level_3": 0.24,
    "dd_hard": 0.275,
    "atr_window": 14,
    "vol_window": 42,
    "fee": TRADING_COST,
}

def sample_configs(space, n, seed):
    rng = np.random.default_rng(seed)
    keys = list(space.keys())
    out = []
    seen = set()

    attempts = 0

    while len(out) < n and attempts < n * 200:
        attempts += 1

        p = {
            k: space[k][
                int(rng.integers(0, len(space[k])))
            ]
            for k in keys
        }

        if p["trend_fast_ema"] >= p["trend_slow_ema"]:
            continue

        if p["btc_fast_days"] >= p["btc_slow_days"]:
            continue

        key = tuple(p[k] for k in keys)

        if key in seen:
            continue

        seen.add(key)
        p.update(fixed_params)
        out.append(p)

    return out


def evaluate_config(p):
    bt = run_strategy_v1(
        market=market,
        btc_market=btc_market,
        start_date=START_DATE,
        **p,
    )

    s = bt["stats"]
    trades = bt["trades"]
    eq = bt["equity"]

    trade_count = len(trades)

    win_rate = (
        (trades["pnl"] > 0).mean()
        if trade_count
        else np.nan
    )

    retest_share = (
        trades["added_on_retest"].mean()
        if trade_count
        and "added_on_retest" in trades.columns
        else np.nan
    )

    avg_trade = (
        trades["return_pct"].mean()
        if trade_count
        else np.nan
    )

    positive_year_share = np.nan
    last30_sharpe = np.nan
    last30_mdd = np.nan

    if len(eq) > 50:

        yearly_ret = (
            eq.groupby(eq.index.year).last()
            / eq.groupby(eq.index.year).first()
            - 1
        )

        if len(yearly_ret):
            positive_year_share = (
                (yearly_ret > 0).mean()
            )

        split_i = int(len(eq) * 0.70)

        if (
            split_i > 30
            and split_i < len(eq) - 30
        ):
            test_eq = eq.iloc[split_i:]
            test_eq = test_eq / test_eq.iloc[0]

            test_stats = performance_stats(test_eq)

            last30_sharpe = test_stats["Sharpe"]
            last30_mdd = test_stats["MDD"]

    return {
        **p,
        **s,
        "Trades": trade_count,
        "WinRate": win_rate,
        "RetestAddShare": retest_share,
        "AvgTrade": avg_trade,
        "PositiveYearShare": positive_year_share,
        "Last30Sharpe": last30_sharpe,
        "Last30MDD": last30_mdd,
    }


def evaluate_many(configs, label):
    rows = []

    for p in tqdm(configs, desc=label):
        rows.append(
            evaluate_config(p)
        )

    return pd.DataFrame(rows)


def clean_results(df):
    x = df.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    x = x.dropna(
        subset=[
            "Sharpe",
            "MDD",
            "CAGR",
        ]
    )

    return x


def get_passes(df):
    x = clean_results(df)

    x = x[x["Trades"] >= 25]

    return x[
        (x["Sharpe"] > 1.0)
        & (x["MDD"] > -0.30)
    ].copy()


def ranking_score(df):
    x = df.copy()

    x["sharpe_shortfall"] = (
        1.0 - x["Sharpe"]
    ).clip(lower=0)

    x["mdd_shortfall"] = (
        -0.30 - x["MDD"]
    ).clip(lower=0)

    x["rank_score"] = (
        1.25 * x["Sharpe"]
        + 0.45 * x["CAGR"].clip(-1, 3)
        + 0.20 * x["Calmar"].fillna(0).clip(-10, 10)
        + 0.10 * x["PositiveYearShare"].fillna(0)
        + 0.08 * x["Last30Sharpe"].fillna(0).clip(-3, 5)
        - 7.0 * x["sharpe_shortfall"]
        - 15.0 * x["mdd_shortfall"]
    )

    return x


stage1_budget = (
    120
    if SEARCH_MODE == "standard"
    else 220
)

stage1_configs = sample_configs(
    base_space,
    stage1_budget,
    RANDOM_SEED,
)

print(
    "Stage-1 broad configurations:",
    len(stage1_configs),
)

results_1 = evaluate_many(
    stage1_configs,
    "V1 broad search",
)

results_1 = clean_results(results_1)

ranked_1 = ranking_score(results_1)

seed_rows = (
    ranked_1
    .sort_values(
        "rank_score",
        ascending=False,
    )
    .head(10)
)

def neighbors_from_seed(row):
    variants = []

    perturb = {
        "breakout_entry_fraction": [-0.10, 0.0, 0.10],
        "risk_per_trade": [-0.004, 0.0, 0.004],
        "position_vol_target": [-0.04, 0.0, 0.04],
        "breakout_buffer_atr": [-0.08, 0.0, 0.08],
        "trailing_stop_atr": [-0.50, 0.0, 0.50],
        "min_rs_percentile": [-0.10, 0.0, 0.10],
    }

    rng = np.random.default_rng(
        RANDOM_SEED + int(row["breakout_window"])
    )

    for _ in range(12):
        p = {
            k: row[k]
            for k in base_space.keys()
        }

        for k, deltas in perturb.items():
            p[k] = float(
                p[k]
                + deltas[
                    int(rng.integers(0, len(deltas)))
                ]
            )

        p["breakout_entry_fraction"] = float(
            np.clip(
                p["breakout_entry_fraction"],
                0.25,
                0.75,
            )
        )

        p["risk_per_trade"] = float(
            np.clip(
                p["risk_per_trade"],
                0.010,
                0.030,
            )
        )

        p["position_vol_target"] = float(
            np.clip(
                p["position_vol_target"],
                0.16,
                0.32,
            )
        )

        p["breakout_buffer_atr"] = float(
            np.clip(
                p["breakout_buffer_atr"],
                0.0,
                0.40,
            )
        )

        p["trailing_stop_atr"] = float(
            np.clip(
                p["trailing_stop_atr"],
                2.5,
                4.5,
            )
        )

        p["min_rs_percentile"] = float(
            np.clip(
                p["min_rs_percentile"],
                0.40,
                0.80,
            )
        )

        for k in [
            "breakout_window",
            "retest_window",
            "trend_fast_ema",
            "trend_slow_ema",
            "btc_fast_days",
            "btc_slow_days",
            "btc_mom_days",
            "time_stop_bars",
            "max_positions",
        ]:
            p[k] = int(p[k])

        p.update(fixed_params)
        variants.append(p)

    return variants


stage2_configs = []

for _, row in seed_rows.iterrows():
    stage2_configs.extend(
        neighbors_from_seed(row)
    )

unique = {}
param_key_cols = list(base_space.keys())

for p in stage2_configs:
    key = tuple(
        p[k]
        for k in param_key_cols
    )
    unique[key] = p

stage2_configs = list(unique.values())

print(
    "Stage-2 local configurations:",
    len(stage2_configs),
)

results_2 = evaluate_many(
    stage2_configs,
    "V1 local refinement",
)

all_results = pd.concat(
    [
        results_1,
        results_2,
    ],
    ignore_index=True,
)

passes = get_passes(all_results)

if AUTO_RESCUE and len(passes) == 0:

    print(
        "\nNo exact PASS after broad + local search. "
        "Running performance-oriented rescue search..."
    )

    rescue_space = dict(base_space)

    rescue_space.update({
        "breakout_entry_fraction": [0.45, 0.55, 0.65, 0.75],
        "min_rs_percentile": [0.50, 0.60, 0.70, 0.80],
        "risk_per_trade": [0.018, 0.022, 0.026, 0.030],
        "position_vol_target": [0.22, 0.26, 0.30, 0.32],
        "max_gross_exposure": [1.00],
        "max_positions": [3, 4],
        "trailing_stop_atr": [3.25, 3.75, 4.25],
        "time_stop_bars": [36, 48, 72],
    })

    rescue_budget = (
        110
        if SEARCH_MODE == "standard"
        else 180
    )

    rescue_configs = sample_configs(
        rescue_space,
        rescue_budget,
        RANDOM_SEED + 9000,
    )

    print(
        "Rescue configurations:",
        len(rescue_configs),
    )

    results_3 = evaluate_many(
        rescue_configs,
        "V1 rescue search",
    )

    all_results = pd.concat(
        [
            all_results,
            results_3,
        ],
        ignore_index=True,
    )

    passes = get_passes(all_results)


results = clean_results(all_results)

if len(passes):

    passes["selection_score"] = (
        1.10 * passes["Sharpe"]
        + 0.55 * passes["CAGR"].clip(-1, 3)
        + 0.20 * passes["Calmar"].fillna(0).clip(-10, 10)
        + 0.10 * passes["PositiveYearShare"].fillna(0)
        + 0.05 * passes["Last30Sharpe"].fillna(0).clip(-3, 5)
    )

    best_row = (
        passes
        .sort_values(
            [
                "selection_score",
                "Sharpe",
                "CAGR",
            ],
            ascending=False,
        )
        .iloc[0]
    )

    print(
        f"\nPASS: {len(passes)} configurations satisfy "
        "Sharpe > 1 and MDD > -30%."
    )

else:

    ranked = ranking_score(
        results[
            results["Trades"] >= 25
        ].copy()
    )

    if ranked.empty:
        ranked = ranking_score(
            results.copy()
        )

    best_row = (
        ranked
        .sort_values(
            [
                "rank_score",
                "Sharpe",
                "MDD",
            ],
            ascending=False,
        )
        .iloc[0]
    )

    print(
        "\nNo sampled V1 configuration passed BOTH constraints. "
        "The notebook reports the best real result; it does not fabricate a pass."
    )


all_param_cols = (
    list(base_space.keys())
    + list(fixed_params.keys())
)

int_cols = {
    "breakout_window",
    "retest_window",
    "trend_fast_ema",
    "trend_slow_ema",
    "btc_fast_days",
    "btc_slow_days",
    "btc_mom_days",
    "time_stop_bars",
    "max_positions",
    "dd_lookback_bars",
    "atr_window",
    "vol_window",
}

best_params = {}

for k in all_param_cols:
    v = best_row[k]

    best_params[k] = (
        int(round(v))
        if k in int_cols
        else float(v)
    )

print("\nBest V1 parameters:")

for k, v in best_params.items():
    print(f"{k}: {v}")


show_cols = [
    "Sharpe",
    "CAGR",
    "Vol",
    "MDD",
    "Calmar",
    "Total Return",
    "Trades",
    "WinRate",
    "RetestAddShare",
    "AvgTrade",
    "PositiveYearShare",
    "Last30Sharpe",
    "Last30MDD",
    "breakout_window",
    "breakout_buffer_atr",
    "breakout_entry_fraction",
    "retest_window",
    "retest_atr",
    "volume_mult",
    "min_rs_percentile",
    "trend_fast_ema",
    "trend_slow_ema",
    "btc_fast_days",
    "btc_slow_days",
    "btc_mom_days",
    "initial_stop_atr",
    "trailing_stop_atr",
    "trail_activation_r",
    "break_even_r",
    "time_stop_bars",
    "max_positions",
    "position_vol_target",
    "risk_per_trade",
    "max_gross_exposure",
]

print("\nTop V1 candidates:")

display(
    results
    .sort_values(
        [
            "Sharpe",
            "MDD",
            "CAGR",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )[show_cols]
    .head(30)
    .style.format({
        "Sharpe": "{:.2f}",
        "CAGR": "{:.2%}",
        "Vol": "{:.2%}",
        "MDD": "{:.2%}",
        "Calmar": "{:.2f}",
        "Total Return": "{:.2%}",
        "WinRate": "{:.2%}",
        "RetestAddShare": "{:.2%}",
        "AvgTrade": "{:.2%}",
        "PositiveYearShare": "{:.0%}",
        "Last30Sharpe": "{:.2f}",
        "Last30MDD": "{:.2%}",
        "breakout_buffer_atr": "{:.2f}",
        "breakout_entry_fraction": "{:.0%}",
        "retest_atr": "{:.2f}",
        "volume_mult": "{:.2f}",
        "min_rs_percentile": "{:.0%}",
        "initial_stop_atr": "{:.2f}",
        "trailing_stop_atr": "{:.2f}",
        "trail_activation_r": "{:.2f}",
        "break_even_r": "{:.2f}",
        "position_vol_target": "{:.0%}",
        "risk_per_trade": "{:.1%}",
        "max_gross_exposure": "{:.0%}",
    })
)


In [9]:
best = run_strategy_v1(
    market=market,
    btc_market=btc_market,
    start_date=START_DATE,
    **best_params,
)

final_stats = best["stats"]

result = pd.DataFrame(
    {
        "Sharpe": [final_stats["Sharpe"]],
        "MDD": [final_stats["MDD"]],
    },
    index=["Strategy V1"],
)

display(
    result.style.format(
        {
            "Sharpe": "{:.2f}",
            "MDD": "{:.2%}",
        }
    )
)

print("Sharpe > 1 :", bool(final_stats["Sharpe"] > 1.0))
print("MDD < 30%  :", bool(abs(final_stats["MDD"]) < 0.30))


,Sharpe,MDD
Strategy V1,1.42,-11.26%


Sharpe > 1 : True
MDD < 30%  : True


In [10]:
final_stats = best["stats"]

result = pd.DataFrame(
    {
        "Total Return": [final_stats["Total Return"]],
        "CAGR": [final_stats["CAGR"]],
        "Vol": [final_stats["Vol"]],
        "Sharpe": [final_stats["Sharpe"]],
        "MDD": [final_stats["MDD"]],
        "Calmar": [final_stats["Calmar"]],
    },
    index=["Strategy V1"],
)

display(
    result.style.format(
        {
            "Total Return": "{:.2%}",
            "CAGR": "{:.2%}",
            "Vol": "{:.2%}",
            "Sharpe": "{:.2f}",
            "MDD": "{:.2%}",
            "Calmar": "{:.2f}",
        }
    )
)

print("Sharpe > 1 :", bool(final_stats["Sharpe"] > 1.0))
print("MDD < 30%  :", bool(abs(final_stats["MDD"]) < 0.30))

,Total Return,CAGR,Vol,Sharpe,MDD,Calmar
Strategy V1,152.82%,20.38%,13.72%,1.42,-11.26%,1.81


Sharpe > 1 : True
MDD < 30%  : True


In [11]:
import plotly.graph_objects as go

equity = best["equity"]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=equity.index,
        y=equity,
        mode="lines",
        name="Equity",
    )
)

fig.update_layout(
    title="Strategy V1 — Equity Curve",
    xaxis_title="Date",
    yaxis_title="Equity",
    hovermode="x unified",
    template="plotly_white",
)

fig.show()